# Read-level variant table — de novo vs COSMIC screen on NA12878 chr20

[`vk_denovo_na12878.ipynb`](vk_denovo_na12878.ipynb) runs varseek over the same NA12878 chr20 reads
twice: once against a reference built from its own **de novo** calls, and once against the whole
**COSMIC** catalog (genotyping mode), scoring the latter against the GIAB truth with `hap.py`.
This notebook joins the two runs at the level of individual reads and produces one table with a row
per variant.

| column | meaning |
| --- | --- |
| `hgvsg` | the variant, HGVS genomic (`20:g.31022441C>T`) — the VCRS header varseek uses |
| `reads_mapped_file1_denovo` / `..._file2_denovo` | read indices (0-based position in the fastq) that pseudoaligned to this variant in the **de novo** run, for R1 and R2 respectively |
| `any_read_mapped_denovo` | whether the de novo run had any read on this variant |
| `reads_mapped_file1_cosmic` / `..._file2_cosmic` / `any_read_mapped_cosmic` | the same for the **COSMIC** screen |
| `tp_cosmic`, `fp_cosmic`, `fn_cosmic` | the `hap.py` verdict on the COSMIC screen: true positive, false positive, and truth variants the screen missed |

Rows are the union of the two runs' variant references, so a variant present in only one of them
still gets a row (with empty read lists on the other side). Nothing is recomputed here except the
read-to-variant bookkeeping — every input is an output of `vk_denovo_na12878.ipynb`.

In [1]:
import collections
import gzip
import json
import os
import re
import subprocess

import numpy as np
import pandas as pd
import pysam
from kb_python.config import get_bustools_binary_path

In [2]:
# ---- inputs: everything here is produced by vk_denovo_na12878.ipynb ----
root = os.path.join("data", "na12878_chr20")
reference_dir = os.path.join("data", "reference")
sequences = os.path.join(reference_dir, "ensembl_grch38_release114", "Homo_sapiens.GRCh38.dna.primary_assembly.fa")

kb_dirs = {
    "denovo": os.path.join(root, "varseek_count_out", "kb_count_out_vcrs"),
    "cosmic": os.path.join(root, "varseek_count_out_cosmic_pseudobam_position_k51_w45", "kb_count_out_vcrs"),
}

happy_vcf = os.path.join(root, "happy_output", "varseek_cosmic_vs_giab_truth_pseudobam_position_k51_w45.vcf.gz")
happy_summary_csv = os.path.join(root, "happy_output", "varseek_cosmic_vs_giab_truth_pseudobam_position_k51_w45.summary.csv")

# vk count ran in bulk "dna" mode, which gives each fastq of the pair its own barcode; the read
# indices in the two files are numbered independently, hence one column per file.
fastqs = {1: os.path.join(root, "reads", "NA12878_chr20_R1.fastq.gz"),
          2: os.path.join(root, "reads", "NA12878_chr20_R2.fastq.gz")}
barcode_to_file = {"AAAAAAAAAAAAAAAA": 1, "AAAAAAAAAAAAAAAC": 2}

min_supporting_reads = 0        # the threshold vk_denovo_na12878.ipynb calls a variant at
work_dir = os.path.join(root, "read_mapping_tmp")
table_out = os.path.join(root, "variant_read_mapping.parquet")
os.makedirs(work_dir, exist_ok=True)

if "counts_unfiltered_pseudobam" in os.listdir(kb_dirs["cosmic"]):
    kb_dirs["cosmic"] = os.path.join(kb_dirs["cosmic"], "counts_unfiltered_pseudobam")

for path in [*kb_dirs.values(), happy_vcf, happy_summary_csv, *fastqs.values()]:
    assert os.path.exists(path), f"missing input: {path}  (run vk_denovo_na12878.ipynb first)"
print("all inputs present")

all inputs present


### The BUS records, and what column 2 actually indexes

`bustools text -f` dumps a BUS file with its flag column, giving one line per pseudoaligned read:

```
barcode                 UMI   equivalence class   count   read index
AAAAAAAAAAAAAAAA        T     0                   1       56310
```

`vk count` runs kallisto with `--num` (its default), which is what puts the read's ordinal
position in its fastq into the flag column, and the barcode identifies which fastq the read came
from. The de novo run's dump was already on disk; the cell below writes the COSMIC one the same way.

**Column 2 is an equivalence class, not a variant.** It is tempting to read it as a row number in
`transcripts.txt`, but kallisto numbers equivalence classes independently of transcripts: in both
runs there are *more* classes than variants, no class id equals the id of the single variant it
contains, and the largest class covers hundreds of variants. Using it as a variant index would
mislabel every multi-variant class — a large share of the records in both runs (printed below) —
and index past the end of `transcripts.txt`. So each class is resolved through `matrix.ec` first,
and the read is credited to **every** variant in its class. That is the same "reads compatible with
a VCRS" convention the recount cell of `vk_denovo_na12878.ipynb` uses to call variants, so the read
lists here agree with the calls scored by `hap.py`.

In [3]:
bustools = str(get_bustools_binary_path())
for tag, kb_dir in kb_dirs.items():
    bus_filename = "output.bus"
    if "counts_unfiltered_pseudobam" in kb_dir:
        bus_filename = "output_filtered.bus"
    bus_txt = os.path.join(kb_dir, "output_bus_with_reads.txt")
    if not os.path.exists(bus_txt):
        print(f"{tag}: converting {bus_filename} to text...")
        subprocess.run([bustools, "text", "-f", "-o", bus_txt, os.path.join(kb_dir, bus_filename)], check=True)
    print(f"{tag}: {bus_txt}")

denovo: data/na12878_chr20/varseek_count_out/kb_count_out_vcrs/output_bus_with_reads.txt
cosmic: data/na12878_chr20/varseek_count_out_cosmic_pseudobam_position_k51_w45/kb_count_out_vcrs/counts_unfiltered_pseudobam/output_bus_with_reads.txt


In [4]:
def load_vcrs_headers(kb_dir):
    """The run's variant reference, in index order. A merged VCRS carries several variants,
    joined with ';' (identical sequences are collapsed into one entry by vk ref)."""
    with open(os.path.join(kb_dir, "transcripts.txt")) as fh:
        return [line.rstrip("\n") for line in fh]


def load_equivalence_classes(kb_dir):
    """matrix.ec as CSR: (offsets, targets) with the variant indices of equivalence class i in
    targets[offsets[i]:offsets[i + 1]]."""
    sizes, chunks = [], []
    with open(os.path.join(kb_dir, "matrix.ec")) as fh:
        for line in fh:
            ec_id, targets = line.rstrip("\n").split("\t")
            assert int(ec_id) == len(sizes), "matrix.ec is not in equivalence-class order"
            arr = np.array(targets.split(","), dtype=np.int32)
            sizes.append(len(arr))
            chunks.append(arr)
    sizes = np.asarray(sizes, dtype=np.int64)
    return np.concatenate([[0], np.cumsum(sizes)]), np.concatenate(chunks)


def load_bus_records(kb_dir):
    """(file index, equivalence class, read index) for every pseudoaligned read.

    Usually straight from the BUS file. A pseudobam-validated recount is the exception: vk clean
    writes its BUS with the per-file barcodes collapsed into the single "good" barcode, so which
    fastq a read came from survives only in the sidecar it writes alongside (one row per kept read,
    with the pre-collapse barcode and the read index).
    """
    provenance = os.path.join(kb_dir, "output_filtered_reads.tsv")
    if os.path.exists(provenance):
        kept = pd.read_csv(provenance, sep="\t", dtype={"raw_barcode": "category",
                                                        "EC": np.int32, "read_index": np.int64})
        categories = list(kept["raw_barcode"].cat.categories)
        unknown = set(categories) - set(barcode_to_file)
        assert not unknown, f"unexpected barcode(s) {unknown}: the fastq mapping needs updating"
        assert (kept["read_index"] >= 0).all(), \
            "the sidecar has read index -1: vk clean could not trace some kept reads to a fastq"
        lookup = np.array([barcode_to_file[c] for c in categories], dtype=np.int8)
        print(f"  read provenance from {os.path.basename(provenance)} (the BUS of a pseudobam-"
              f"validated recount carries the collapsed barcode, so it cannot give the fastq)")
        return (lookup[kept["raw_barcode"].cat.codes.to_numpy()], kept["EC"].to_numpy(),
                kept["read_index"].to_numpy())

    bus = pd.read_csv(os.path.join(kb_dir, "output_bus_with_reads.txt"), sep="\t", header=None,
                      names=["barcode", "umi", "ec", "count", "read"],
                      dtype={"barcode": "category", "ec": np.int32,
                             "count": np.int32, "read": np.int64})
    assert (bus["count"] == 1).all(), "expected exactly one read per BUS record (kallisto --num)"
    categories = list(bus["barcode"].cat.categories)
    unknown = set(categories) - set(barcode_to_file)
    assert not unknown, f"unexpected barcode(s) {unknown}: the fastq mapping needs updating"
    lookup = np.array([barcode_to_file[c] for c in categories], dtype=np.int8)
    return lookup[bus["barcode"].cat.codes.to_numpy()], bus["ec"].to_numpy(), bus["read"].to_numpy()


def ragged_indices(starts, sizes):
    """Flat indices that take sizes[i] consecutive values starting at starts[i], for all i."""
    total = int(sizes.sum())
    within = np.arange(total, dtype=np.int64) - np.repeat(np.cumsum(sizes) - sizes, sizes)
    return np.repeat(starts, sizes) + within


def group_read_lists(rows, reads, n_rows):
    """Per-row sorted list of distinct read indices (a read reaching one variant through two
    merged VCRSs is one read, not two)."""
    order = np.lexsort((reads, rows))
    rows, reads = rows[order], reads[order]
    if len(rows):
        keep = np.empty(len(rows), dtype=bool)
        keep[0] = True
        keep[1:] = (rows[1:] != rows[:-1]) | (reads[1:] != reads[:-1])
        rows, reads = rows[keep], reads[keep]
    counts = np.bincount(rows, minlength=n_rows)
    return [chunk.tolist() for chunk in np.split(reads, np.cumsum(counts)[:-1])], counts


def variant_read_lists(kb_dir, row_of_variant, n_rows):
    """Reads per variant for one varseek run: {file index: [[read, ...], ...]} aligned to the
    table's rows, plus the per-VCRS read count that vk count thresholds to call variants."""
    headers = load_vcrs_headers(kb_dir)
    ec_offsets, ec_targets = load_equivalence_classes(kb_dir)
    ec_sizes = np.diff(ec_offsets)
    file_index, ec, read = load_bus_records(kb_dir)
    print(f"  {len(ec):,} BUS records over {len(ec_sizes):,} equivalence classes and "
          f"{len(headers):,} variants")
    print(f"  {100 * np.mean(ec_sizes[ec] > 1):5.1f}% of records are in a multi-variant class "
          f"(largest class: {ec_sizes.max():,} variants)")

    # 1) equivalence class -> VCRS
    sizes = ec_sizes[ec]
    take = ragged_indices(ec_offsets[ec], sizes)
    vcrs = ec_targets[take].astype(np.int64)
    file_index, read = np.repeat(file_index, sizes), np.repeat(read, sizes)
    vcrs_counts = np.bincount(vcrs, minlength=len(headers))

    # 2) VCRS -> table row(s); a merged VCRS credits its reads to each variant it carries
    row_sizes = np.empty(len(headers), dtype=np.int64)
    rows_flat = []
    for i, header in enumerate(headers):
        parts = [p for p in header.split(";") if p]
        row_sizes[i] = len(parts)
        rows_flat.extend(row_of_variant[p] for p in parts)
    row_offsets = np.concatenate([[0], np.cumsum(row_sizes)])
    rows_flat = np.asarray(rows_flat, dtype=np.int64)

    sizes = row_sizes[vcrs]
    take = ragged_indices(row_offsets[vcrs], sizes)
    rows = rows_flat[take]
    file_index, read = np.repeat(file_index, sizes), np.repeat(read, sizes)
    print(f"  {len(rows):,} (variant, read) assignments after expanding classes and merged VCRSs")

    # Can this run's reads still be traced back to the fastqs? kb count's --num puts a read's
    # position in its fastq into the BUS flag column, and the barcode says which fastq it came from.
    # A BUS that was rewritten downstream can lose both: vk clean's pseudobam-validated recount
    # collapses the per-file barcodes into one, and -- before varseek carried the read index through
    # (`_recount_from_filtered_bus`) -- wrote every record with flag 0. The read lists then say
    # "read 0 of file 1" for every variant, which is a placeholder, not a read.
    provenance = {"files": [int(f) for f in np.unique(file_index)],
                  "distinct_read_indices": int(np.unique(read).size)}
    provenance["usable"] = provenance["distinct_read_indices"] > 1
    if not provenance["usable"]:
        print(f"  !! no read indices: every BUS record carries flag {int(np.unique(read)[0])}, so this "
              f"run's read columns are a placeholder rather than reads")
    if len(provenance["files"]) < len(fastqs):
        print(f"  !! only fastq file(s) {provenance['files']} are represented: the per-file barcodes "
              f"were collapsed, so which mate a read came from is no longer recorded")

    lists = {}
    for f in sorted(fastqs):
        mask = file_index == f
        lists[f], _ = group_read_lists(rows[mask], read[mask], n_rows)
    return lists, vcrs_counts, headers, provenance

In [5]:
# Rows = every variant either run could have reported, in de novo-then-COSMIC order.
row_of_variant = {}
headers_by_run = {}
for tag, kb_dir in kb_dirs.items():
    headers_by_run[tag] = load_vcrs_headers(kb_dir)
    for header in headers_by_run[tag]:
        for variant in header.split(";"):
            if variant and variant not in row_of_variant:
                row_of_variant[variant] = len(row_of_variant)

n_rows = len(row_of_variant)
hgvsg = np.empty(n_rows, dtype=object)
for variant, row in row_of_variant.items():
    hgvsg[row] = variant
print(f"variants in the de novo reference : {sum(h.count(';') + 1 for h in headers_by_run['denovo']):,}")
print(f"variants in the COSMIC reference  : {sum(h.count(';') + 1 for h in headers_by_run['cosmic']):,}")
print(f"union (rows in the table)         : {n_rows:,}")

variants in the de novo reference : 102,066
variants in the COSMIC reference  : 1,495,338
union (rows in the table)         : 1,555,949


In [6]:
read_lists, vcrs_counts, read_provenance = {}, {}, {}
for tag, kb_dir in kb_dirs.items():
    print(f"{tag}:")
    read_lists[tag], vcrs_counts[tag], _, read_provenance[tag] = variant_read_lists(
        kb_dir, row_of_variant, n_rows)

denovo:
  2,562,263 BUS records over 315,955 equivalence classes and 101,804 variants
   43.0% of records are in a multi-variant class (largest class: 197 variants)
  15,044,508 (variant, read) assignments after expanding classes and merged VCRSs
cosmic:
  read provenance from output_filtered_reads.tsv (the BUS of a pseudobam-validated recount carries the collapsed barcode, so it cannot give the fastq)
  1,991 BUS records over 137,576 equivalence classes and 1,453,397 variants
   30.8% of records are in a multi-variant class (largest class: 123 variants)
  8,380 (variant, read) assignments after expanding classes and merged VCRSs


### Carrying the `hap.py` verdicts back to the variants

`hap.py` works in VCF space and `hap.py`'s own output VCF is the authority on which call was a TP,
an FP, or a missed truth variant (`FORMAT/BD` on the QUERY and TRUTH samples). Getting those
verdicts onto HGVS rows means undoing the HGVS → VCF → left-normalization path the benchmark took:

* **query side (TP / FP)** — the variants the COSMIC screen called are re-emitted as a VCF whose
  `ID` field carries the HGVS string, pushed through the *same* `bcftools norm` the benchmark used,
  and joined to `hap.py`'s records on `CHROM/POS/REF/ALT`;
* **truth side (FN)** — the same, built from the COSMIC catalog, since the truth set of that
  benchmark is GIAB restricted to COSMIC and every missed truth variant is a catalog entry.

Both maps are asserted to be one-to-one and to resolve *every* labelled `hap.py` record, so the
verdicts land on exactly the variants they were computed for. (Building one shared map over both
runs' variants would not be safe: de novo and COSMIC indels in short repeats can normalize onto the
same coordinates from different HGVS strings, and the verdict would then be ambiguous.)

In [7]:
CHROMOSOMES = {str(i) for i in range(1, 23)} | {"X", "Y", "MT"}


def hgvs_to_vcf_frame(variants, fasta):
    """HGVS genomic -> CHROM/POS/REF/ALT/HGVS, with the grammar `adata_to_vcf` uses in
    vk_denovo_na12878.ipynb, so these records land on the coordinates hap.py scored."""
    s = pd.Series(list(variants), dtype="object")
    snv = s.str.extract(r"^(?P<CHROM>.+):g\.(?P<POS>\d+)(?P<REF>[ACGT]+)>(?P<ALT>[ACGT]+)$")
    ins = s.str.extract(r"^(?P<CHROM>.+):g\.(?P<POS>\d+)_(?P<END>\d+)ins(?P<INS>[ACGT]+)$")
    dele = s.str.extract(r"^(?P<CHROM>.+):g\.(?P<START>\d+)(?:_(?P<END>\d+))?del(?P<DEL>[ACGT]*)$")
    snv_mask, ins_mask, del_mask = snv["CHROM"].notna(), ins["CHROM"].notna(), dele["START"].notna()
    n_unparsed = int((~(snv_mask | ins_mask | del_mask)).sum())

    frames = []
    part = snv[snv_mask & snv["CHROM"].isin(CHROMOSOMES)]
    if not part.empty:
        frames.append(pd.DataFrame({"CHROM": part["CHROM"], "POS": part["POS"].astype(int),
                                    "REF": part["REF"], "ALT": part["ALT"], "HGVS": s[part.index]}))
    part = ins[ins_mask & ins["CHROM"].isin(CHROMOSOMES)]
    if not part.empty:
        pos = part["POS"].astype(int)
        anchor = pd.Series([fasta.fetch(c, p - 1, p) for c, p in zip(part["CHROM"], pos)],
                           index=part.index)
        frames.append(pd.DataFrame({"CHROM": part["CHROM"], "POS": pos, "REF": anchor,
                                    "ALT": anchor + part["INS"], "HGVS": s[part.index]}))
    part = dele[del_mask & dele["CHROM"].isin(CHROMOSOMES)]
    if not part.empty:
        start = part["START"].astype(int)
        end = pd.to_numeric(part["END"]).fillna(start).astype(int)
        anchor = pd.Series([fasta.fetch(c, p - 2, p - 1) for c, p in zip(part["CHROM"], start)],
                           index=part.index)
        deleted = pd.Series([d if isinstance(d, str) and d != "" else fasta.fetch(c, s0 - 1, e0)
                             for c, s0, e0, d in zip(part["CHROM"], start, end, part["DEL"])],
                            index=part.index)
        frames.append(pd.DataFrame({"CHROM": part["CHROM"], "POS": start - 1,
                                    "REF": anchor + deleted, "ALT": anchor, "HGVS": s[part.index]}))
    return pd.concat(frames, ignore_index=True), n_unparsed


def normalized_key_map(variants, label):
    """{(chrom, pos, ref, alt) -> hgvs} after the benchmark's left-normalization."""
    frame, n_unparsed = hgvs_to_vcf_frame(variants, pysam.FastaFile(sequences))
    frame = frame.drop_duplicates(subset=["CHROM", "POS", "REF", "ALT", "HGVS"])
    sites = os.path.join(work_dir, f"{label}_with_ids.vcf")
    with open(sites, "w") as fh:
        fh.write("##fileformat=VCFv4.2\n#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
        fh.write("".join(frame["CHROM"].astype(str) + "\t" + frame["POS"].astype(str) + "\t"
                         + frame["HGVS"] + "\t" + frame["REF"] + "\t" + frame["ALT"] + "\t.\t.\t.\n"))

    normalized = sites.replace(".vcf", ".normalized.vcf.gz")
    fai = f"{sequences}.fai"
    if not os.path.isfile(fai):
        subprocess.run(["samtools", "faidx", sequences], check=True)
    reheader = subprocess.Popen(["bcftools", "reheader", "--fai", fai, sites], stdout=subprocess.PIPE)
    norm = subprocess.Popen(["bcftools", "norm", "-c", "w", "-f", sequences, "-m", "-both", "-Ou"],
                            stdin=reheader.stdout, stdout=subprocess.PIPE)
    reheader.stdout.close()
    subprocess.run(["bcftools", "sort", "-Oz", "-o", normalized], stdin=norm.stdout, check=True)
    norm.stdout.close(), reheader.wait(), norm.wait()

    key_map = collections.defaultdict(list)
    with gzip.open(normalized, "rt") as fh:
        for line in fh:
            if line.startswith("#"):
                continue
            c = line.split("\t")
            key_map[(c[0], int(c[1]), c[3], c[4])].append(c[2])

    # A few coordinates carry more than one HGVS spelling, which is not an error: an indel inside a
    # repeat can be written at several equivalent positions (12:g.49031691_49031694del and
    # 12:g.49031696_49031699del are the same 4 bp deletion), and left-normalization is precisely
    # what collapses those spellings onto one record. Fold each group onto one representative, so a
    # hap.py record still resolves to exactly one variant and the record totals stay reconcilable.
    # The folded-away spellings keep their own rows in the table, carrying no verdict.
    aliased = {k: sorted(set(v)) for k, v in key_map.items() if len(set(v)) > 1}
    if aliased:
        example = "; ".join(f"{k[0]}:{k[1]} <- {' = '.join(v)}" for k, v in list(aliased.items())[:2])
        print(f"{label}: {len(aliased):,} coordinates have >1 HGVS spelling of the same variant "
              f"({sum(len(v) - 1 for v in aliased.values()):,} folded onto a representative), "
              f"e.g. {example}")
    print(f"{label}: {len(frame):,} variants -> {len(key_map):,} normalized coordinates "
          f"({n_unparsed:,} HGVS types not expressible as VCF: dup/delins/inv)")
    return {k: sorted(set(v))[0] for k, v in key_map.items()}

In [8]:
# Variants the COSMIC screen called -- the query side of the benchmark.
called_cosmic = sorted({variant
                        for header, count in zip(headers_by_run["cosmic"], vcrs_counts["cosmic"])
                        if count >= min_supporting_reads
                        for variant in header.split(";") if variant})
query_map = normalized_key_map(called_cosmic, "cosmic_calls")

# The catalog, for the truth side: the benchmark's truth is GIAB restricted to COSMIC, so every
# missed truth variant is a chr20 catalog entry.
catalog_chr20 = sorted({variant for header in headers_by_run["cosmic"]
                        for variant in header.split(";") if variant.startswith("20:")})
catalog_map = normalized_key_map(catalog_chr20, "cosmic_catalog")

Writing to /home/jrich/tmp/bcftools.f38wog
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	1410486/0/0/67777/0/0/0
Merging 1 temporary files
Done
Cleaning


cosmic_calls: 10 coordinates have >1 HGVS spelling of the same variant (10 folded onto a representative), e.g. 12:49031690 <- 12:g.49031691_49031694del = 12:g.49031696_49031699del; 13:28034086 <- 13:g.28034133_28034134insTCTGAAATCTAAATTTTCTCTTGGAAACTCCCATTTGAGATCATATTCATATTC = 13:g.28034135_28034136insTGAAATCTAAATTTTCTCTTGGAAACTCCCATTTGAGATCATATTCATATTCTC
cosmic_calls: 1,410,486 variants -> 1,410,476 normalized coordinates (43,534 HGVS types not expressible as VCF: dup/delins/inv)


Writing to /home/jrich/tmp/bcftools.nJfFnq
Lines   total/split/joined/realigned/mismatch_removed/dup_removed/skipped:	30177/0/0/1113/0/0/0


cosmic_catalog: 30,177 variants -> 30,177 normalized coordinates (691 HGVS types not expressible as VCF: dup/delins/inv)


Merging 1 temporary files
Done
Cleaning


In [9]:
# hap.py's annotated VCF: FORMAT/BD on the TRUTH sample (col 10) and the QUERY sample (col 11).
records = {"tp": collections.Counter(), "fp": collections.Counter(), "fn": collections.Counter()}
unresolved = collections.Counter()

with gzip.open(happy_vcf, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        keys = fields[8].split(":")
        truth = dict(zip(keys, fields[9].split(":")))
        query = dict(zip(keys, fields[10].split(":")))
        key = (fields[0], int(fields[1]), fields[3], fields[4])

        if query.get("BD") in ("TP", "FP"):          # UNK = outside the confident regions
            variant = query_map.get(key)
            if variant is None:
                unresolved[f"query {query['BD']}"] += 1
            else:
                records[query["BD"].lower()][variant] += 1
        if truth.get("BD") == "FN":
            variant = catalog_map.get(key)
            if variant is None:
                unresolved["truth FN"] += 1
            else:
                records["fn"][variant] += 1

assert not unresolved, f"hap.py records that could not be mapped back to a variant: {dict(unresolved)}"

summary = pd.read_csv(happy_summary_csv)
summary = summary[summary["Filter"] == "PASS"].set_index("Type")
expected = {"tp": int(summary["TRUTH.TP"].sum()), "fp": int(summary["QUERY.FP"].sum()),
            "fn": int(summary["TRUTH.FN"].sum())}
for label, counter in records.items():
    n_records, n_variants = sum(counter.values()), len(counter)
    print(f"{label.upper()}: {n_records:>4} hap.py records -> {n_variants:>4} distinct variants "
          f"(hap.py summary: {expected[label]})")
    assert n_records == expected[label], f"{label}: {n_records} records vs {expected[label]} in the hap.py summary"

duplicated = {label: sum(v - 1 for v in counter.values() if v > 1) for label, counter in records.items()}
print("\nrecords beyond one per variant (the same call listed twice in the query VCF):", duplicated)
assert all(variant not in records["fn"] for variant in records["fp"]), "a variant is both FP and FN"

TP:   85 hap.py records ->   85 distinct variants (hap.py summary: 85)
FP:   23 hap.py records ->   23 distinct variants (hap.py summary: 23)
FN:   57 hap.py records ->   57 distinct variants (hap.py summary: 57)

records beyond one per variant (the same call listed twice in the query VCF): {'tp': 0, 'fp': 0, 'fn': 0}


In [10]:
def lengths(lists):
    return np.fromiter((len(x) for x in lists), dtype=np.int64, count=n_rows)


def flags(counter):
    """(bool per row, hap.py record count per row) for one verdict."""
    counts = np.zeros(n_rows, dtype=np.int32)
    for variant, n in counter.items():
        counts[row_of_variant[variant]] = n
    return counts > 0, counts


tp, tp_records = flags(records["tp"])
fp, fp_records = flags(records["fp"])
fn, fn_records = flags(records["fn"])

df = pd.DataFrame({
    "hgvsg": hgvsg,
    "reads_mapped_file1_denovo": read_lists["denovo"][1],
    "reads_mapped_file2_denovo": read_lists["denovo"][2],
    "any_read_mapped_denovo": (lengths(read_lists["denovo"][1])
                               + lengths(read_lists["denovo"][2])) > 0,
    "reads_mapped_file1_cosmic": read_lists["cosmic"][1],
    "reads_mapped_file2_cosmic": read_lists["cosmic"][2],
    "any_read_mapped_cosmic": (lengths(read_lists["cosmic"][1])
                               + lengths(read_lists["cosmic"][2])) > 0,
    "tp_cosmic": tp,
    "fp_cosmic": fp,
    "fn_cosmic": fn,
    # hap.py counts records, not variants; these keep the totals reconcilable (see the checks)
    "tp_records_cosmic": tp_records,
    "fp_records_cosmic": fp_records,
    "fn_records_cosmic": fn_records,
})
print(f"{len(df):,} rows x {df.shape[1]} columns")
df.head()

1,555,949 rows x 13 columns


,hgvsg,reads_mapped_file1_denovo,reads_mapped_file2_denovo,any_read_mapped_denovo,reads_mapped_file1_cosmic,reads_mapped_file2_cosmic,any_read_mapped_cosmic,tp_cosmic,fp_cosmic,fn_cosmic,tp_records_cosmic,fp_records_cosmic,fn_records_cosmic
0,20:g.80457C>T,"[61611, 1808875, 5344499]","[948917, 1624099, 1636763, 1682487, 2727929, 3...",True,[],[],False,False,False,False,0,0,0
1,20:g.80497_80498insT,"[1114781, 1808875, 2517710, 3345305]","[1624099, 1636763, 2607675, 2727929]",True,[],[],False,False,False,False,0,0,0
2,20:g.81154T>G,"[915701, 1328618, 1742124, 2988576, 3947509, 5...","[1019175, 1044643, 2318211, 3080779, 3237693, ...",True,[],[],False,False,False,False,0,0,0
3,20:g.82603A>C,"[243382, 625446, 3481978, 3550800, 3915818, 40...","[160136, 500998, 1109600, 2278518, 3025813, 39...",True,[],[],False,False,False,False,0,0,0
4,20:g.83158C>T,"[405343, 2855057, 3025813, 3763676, 3765342, 4...","[243382, 3753173, 3829946, 5276947]",True,[],[],False,False,False,False,0,0,0


In [11]:
# ---- consistency checks -------------------------------------------------------------------
# 1) the record columns reproduce the hap.py confusion matrix exactly
for label, column in (("tp", "tp_records_cosmic"), ("fp", "fp_records_cosmic"), ("fn", "fn_records_cosmic")):
    assert int(df[column].sum()) == expected[label], column
print(f"hap.py totals   TP={expected['tp']}  FP={expected['fp']}  FN={expected['fn']}")
print(f"table totals    TP={df.tp_records_cosmic.sum()}  FP={df.fp_records_cosmic.sum()}  "
      f"FN={df.fn_records_cosmic.sum()}   (records)")
print(f"                TP={df.tp_cosmic.sum()}  FP={df.fp_cosmic.sum()}  "
      f"FN={df.fn_cosmic.sum()}   (distinct variants)")

# 2) the two error columns never land on the same variant -- a variant is either wrongly called
#    or wrongly missed, not both.
assert not (df.fp_cosmic & df.fn_cosmic).any(), "a variant is flagged as both FP and FN"

# A few variants do legitimately carry two verdicts, because hap.py labels VCF *records* while
# this table is per variant. Both causes are visible in hap.py's output VCF:
#   * the query VCF holds the same call twice -- vk ref can emit a merged VCRS header that
#     repeats a variant ("20:g.51790963A>C;20:g.51790963A>C"), and adata_to_vcf writes one record
#     per copy, so hap.py matches one to truth (TP) and has nothing left to match the other (FP);
#   * an overlapping half-call locus, where the truth allele on one haplotype is matched (TP) and
#     the one on the other is not (FN).
# They are kept exactly as hap.py assigned them, which is what keeps the totals above exact.
both = df[(df.tp_cosmic & df.fp_cosmic) | (df.tp_cosmic & df.fn_cosmic)]
print(f"\nvariants carrying two hap.py verdicts: {len(both)}")
display(both[["hgvsg", "tp_cosmic", "fp_cosmic", "fn_cosmic",
              "tp_records_cosmic", "fp_records_cosmic", "fn_records_cosmic"]])

# 3) every TP/FP variant really did collect the reads that got it called -- the end-to-end check
#    that the read lists and the benchmark agree
cosmic_reads = (df["reads_mapped_file1_cosmic"].str.len() + df["reads_mapped_file2_cosmic"].str.len())
called = df.tp_cosmic | df.fp_cosmic
assert (cosmic_reads[called] >= min_supporting_reads).all(), \
    "a called variant has fewer reads than the calling threshold"
print(f"\nevery one of the {int(called.sum()):,} TP/FP variants carries >= {min_supporting_reads} "
      f"COSMIC-run reads (min {int(cosmic_reads[called].min())}, median "
      f"{int(cosmic_reads[called].median())}, max {int(cosmic_reads[called].max()):,})")

# The buckets are exclusive, so they have to add up to the FN count -- at min_supporting_reads=0
# "under the threshold" is empty and every variant with a read is in the last bucket.
fn_reads = cosmic_reads[df.fn_cosmic]
called_from = max(min_supporting_reads, 1)
print(f"the {int(df.fn_cosmic.sum())} missed truth variants split into "
      f"{int((fn_reads == 0).sum())} with no read at all, "
      f"{int(((fn_reads > 0) & (fn_reads < called_from)).sum())} with 1-{called_from - 1} "
      f"reads (under the calling threshold), and "
      f"{int((fn_reads >= called_from).sum())} at or above it (the half-call locus listed "
      f"above, which hap.py also counts as a TP)")

hap.py totals   TP=85  FP=23  FN=57
table totals    TP=85  FP=23  FN=57   (records)
                TP=85  FP=23  FN=57   (distinct variants)

variants carrying two hap.py verdicts: 2


,hgvsg,tp_cosmic,fp_cosmic,fn_cosmic,tp_records_cosmic,fp_records_cosmic,fn_records_cosmic
65843,20:g.45325634_45325635insT,True,True,False,1,1,0
76897,20:g.51790963A>C,True,True,False,1,1,0



every one of the 106 TP/FP variants carries >= 0 COSMIC-run reads (min 3, median 6, max 34)
the 57 missed truth variants split into 40 with no read at all, 0 with 1-0 reads (under the calling threshold), and 17 at or above it (the half-call locus listed above, which hap.py also counts as a TP)


In [12]:
# ---- what the table contains --------------------------------------------------------------
in_reference = {}
for tag, headers in headers_by_run.items():
    flag = np.zeros(n_rows, dtype=bool)
    for header in headers:
        for variant in header.split(";"):
            if variant:
                flag[row_of_variant[variant]] = True
    in_reference[tag] = flag

denovo_hit, cosmic_hit = df["any_read_mapped_denovo"], df["any_read_mapped_cosmic"]
pairs = {tag: int(sum(lengths(read_lists[tag][f]).sum() for f in fastqs)) for tag in kb_dirs}

print(f"rows (variants)                      : {len(df):,}")
print(f"  in the de novo reference           : {int(in_reference['denovo'].sum()):,}")
print(f"  in the COSMIC reference            : {int(in_reference['cosmic'].sum()):,}")
print(f"  in both references                 : {int((in_reference['denovo'] & in_reference['cosmic']).sum()):,}")
print(f"with >=1 read, de novo run           : {int(denovo_hit.sum()):,}")
print(f"with >=1 read, COSMIC run            : {int(cosmic_hit.sum()):,}")
print(f"with >=1 read in both runs           : {int((denovo_hit & cosmic_hit).sum()):,}")
print(f"(variant, read) pairs, de novo run   : {pairs['denovo']:,}")
print(f"(variant, read) pairs, COSMIC run    : {pairs['cosmic']:,}")

print("\nexample rows -- variants the COSMIC screen got right:")
display(df[df.tp_cosmic].head(5))

rows (variants)                      : 1,555,949
  in the de novo reference           : 102,066
  in the COSMIC reference            : 1,454,020
  in both references                 : 137
with >=1 read, de novo run           : 96,503
with >=1 read, COSMIC run            : 3,001
with >=1 read in both runs           : 97
(variant, read) pairs, de novo run   : 15,044,508
(variant, read) pairs, COSMIC run    : 8,208

example rows -- variants the COSMIC screen got right:


,hgvsg,reads_mapped_file1_denovo,reads_mapped_file2_denovo,any_read_mapped_denovo,reads_mapped_file1_cosmic,reads_mapped_file2_cosmic,any_read_mapped_cosmic,tp_cosmic,fp_cosmic,fn_cosmic,tp_records_cosmic,fp_records_cosmic,fn_records_cosmic
3863,20:g.1915196T>C,"[296602, 1150381, 3350379, 3811815, 4349594, 4...","[158005, 706957, 933950, 1221388, 1632093, 193...",True,[3350379],"[3549132, 3693401, 4353234, 4894119, 6149101, ...",True,True,False,False,1,0,0
3875,20:g.1915454C>T,"[311104, 600043, 685775, 1249007, 1554805, 253...","[692962, 1272140, 2078580, 2625947, 2982511, 3...",True,[4964759],"[692962, 1272140, 5764010]",True,True,False,False,1,0,0
3876,20:g.1915598A>G,"[277734, 424559, 600043, 606663, 706957, 13875...","[141618, 1251362, 1258103, 1845975, 2155619, 2...",True,"[277734, 424559, 706957, 1523233, 2447806, 361...","[1251362, 1845975, 5841917]",True,True,False,False,1,0,0
3909,20:g.1934580G>A,"[1815494, 2149690, 2666665, 3801205, 5063500, ...","[337048, 520664, 668147, 690770, 2534668, 3059...",True,"[1815494, 2149690]","[668147, 2534668, 3059386, 4607983]",True,True,False,False,1,0,0
37142,20:g.20052354T>C,"[90480, 1862655, 1894922, 2082379, 4580604, 58...","[82265, 1894446, 1978113, 2062296, 4332192, 44...",True,"[1862655, 5827557]","[1978113, 2062296, 5131055, 6555349]",True,True,False,False,1,0,0


In [13]:
# ---- spot-check the read indices against the fastq files ------------------------------------
# The strongest check available on the read column: rebuild the sequence a VCRS is made of (the
# reference window with the variant substituted) and confirm that the reads this table lists for
# that variant really do share a k-mer with it. kallisto only puts a variant in a read's
# equivalence class when they share a k-mer, so this has to hold for every listed read.
# The COSMIC run is checked whenever its reads can still be traced back to the fastqs; when they
# cannot (see the provenance warning printed above) the de novo run is checked instead. Both read
# columns are built by the same code, so the check is equally informative on either run.
spot_tag = "cosmic" if read_provenance["cosmic"]["usable"] else "denovo"


def kb_count_k(kb_dir):
    """The k kb count ran with. A pseudobam-validated recount directory has no kb_info.json of its
    own -- it inherits the settings of the run one level up that it was derived from."""
    for directory in (kb_dir, os.path.dirname(kb_dir)):
        info = os.path.join(directory, "kb_info.json")
        if os.path.exists(info):
            with open(info) as fh:
                return int(re.search(r"-k (\d+)", json.load(fh)["call"]).group(1))
    raise FileNotFoundError(f"no kb_info.json in {kb_dir} or its parent")


k = kb_count_k(kb_dirs["cosmic"])
k_spot = kb_count_k(kb_dirs[spot_tag])
flank = k_spot - 1   # vk ref was run with w = k - 1


def fastq_sequences(path, wanted):
    """{read index: sequence} for 0-based read positions in a fastq."""
    wanted, found = set(wanted), {}
    with gzip.open(path, "rt") as fh:
        for i, _header in enumerate(fh):
            seq = next(fh)
            next(fh), next(fh)
            if i in wanted:
                found[i] = seq.strip()
                if len(found) == len(wanted):
                    break
    return found


def reverse_complement(seq):
    return seq.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]


def snv_vcrs_sequence(variant, fasta):
    """The VCRS sequence of an SNV: `flank` bp of reference either side of the substituted base."""
    chrom, pos, ref, alt = re.match(r"^(.+):g\.(\d+)([ACGT]+)>([ACGT]+)$", variant).groups()
    pos = int(pos)
    assert fasta.fetch(chrom, pos - 1, pos) == ref, "reference allele mismatch"
    return fasta.fetch(chrom, pos - 1 - flank, pos - 1) + alt + fasta.fetch(chrom, pos, pos + flank)


# An SNV that sits alone in its VCRS (so every read on it is there for this variant and not for a
# neighbour merged into the same sequence), picked to have its reads early in the fastqs so the scan
# stays short. Reads are asked for from both fastqs, dropping to whatever the run has if a variant
# with five of them in each does not exist.
solo = {h for h in headers_by_run[spot_tag] if ";" not in h}
snvs = df[df.hgvsg.str.contains(">") & df.hgvsg.isin(solo)]
if spot_tag == "cosmic":
    snvs = snvs[snvs.tp_cosmic]      # ... and one hap.py scored as a true positive
reads_1, reads_2 = snvs[f"reads_mapped_file1_{spot_tag}"], snvs[f"reads_mapped_file2_{spot_tag}"]

for n_check in (5, 3, 2, 1):
    candidates = [(max(sorted(r1)[:n_check] + sorted(r2)[:n_check]), v, sorted(r1)[:n_check], sorted(r2)[:n_check])
                  for v, r1, r2 in zip(snvs.hgvsg, reads_1, reads_2)
                  if len(r1) >= n_check and len(r2) >= n_check]
    if candidates:
        break
assert candidates, (f"no solo SNV of the {spot_tag} run carries a read in both fastqs, so the read "
                    f"columns cannot be checked against the reads themselves")
_, variant, reads1, reads2 = min(candidates)
vcrs = snv_vcrs_sequence(variant, pysam.FastaFile(sequences))
print(f"checking {variant} of the {spot_tag} run, {n_check} read(s) per fastq  "
      f"(k={k_spot}, flank={flank})\nVCRS sequence: {vcrs}\n")

for file_index, reads in ((1, reads1), (2, reads2)):
    by_read = fastq_sequences(fastqs[file_index], reads)
    for read in reads:
        seq = by_read[read]
        kmers = {seq[i:i + k_spot] for i in range(len(seq) - k_spot + 1)}
        shared = sum(1 for kmer in kmers if kmer in vcrs or reverse_complement(kmer) in vcrs)
        assert shared, f"read {read} of file{file_index} shares no {k_spot}-mer with the VCRS"
        print(f"  file{file_index} read {read:>9}: {shared:>3} shared {k_spot}-mers  ok")
print("\nthe read indices resolve to reads that really do carry the variant")

checking 20:g.41041909T>G of the cosmic run, 5 read(s) per fastq  (k=51, flank=50)
VCRS sequence: CACAGGTTAAGGTATAAAGCTTACAGGGTGATGATGATGATGATGATGATGATTATTATTATTATTTGGAGACAGAGTCTCCCTTTGTCGCCCAGGCTGGG

  file1 read    358057:   6 shared 51-mers  ok
  file1 read   1738772:  11 shared 51-mers  ok
  file1 read   2143171:  35 shared 51-mers  ok
  file1 read   2373485:  20 shared 51-mers  ok
  file1 read   2615893:  50 shared 51-mers  ok
  file2 read    691996:  45 shared 51-mers  ok
  file2 read    781208:  24 shared 51-mers  ok
  file2 read    851739:  11 shared 51-mers  ok
  file2 read   1738772:  24 shared 51-mers  ok
  file2 read   4512280:  51 shared 51-mers  ok

the read indices resolve to reads that really do carry the variant


In [14]:
df.to_parquet(table_out, index=False)
print(f"wrote {len(df):,} rows to {table_out} "
      f"({os.path.getsize(table_out) / 1e6:.1f} MB)")
print("\nread it back with:  pd.read_parquet(table_out)")

wrote 1,555,949 rows to data/na12878_chr20/variant_read_mapping.parquet (86.0 MB)

read it back with:  pd.read_parquet(table_out)


### Notes on how to read the table

* **Read indices are per file.** `reads_mapped_file1_*` indexes `NA12878_chr20_R1.fastq.gz` and
  `reads_mapped_file2_*` indexes `NA12878_chr20_R2.fastq.gz`, both 0-based and numbered
  independently — the two files are mates, so the same index in both columns is one read pair.
* **A read can support several variants.** Reads are credited to every variant in their
  equivalence class, so the read lists overlap between variants that share a k-mer. This is the
  convention the benchmark's calls were made under, not an artefact of this table.
* **`tp/fp/fn_cosmic` only mean anything on chr20, inside the GIAB confident regions.** That is
  where the benchmark is defined; calls elsewhere (including everything off chr20 in the
  genome-wide COSMIC reference) are `UNK` to `hap.py` and are all-`False` here.
* **Records vs variants.** `hap.py` counts VCF *records*: two of its false positives are the same
  variant listed twice, because a variant that sits in two merged VCRSs is written to the query VCF
  twice. The `*_records_cosmic` columns preserve `hap.py`'s totals exactly, while the boolean
  columns are per distinct variant — hence 812 FP records over 810 FP variants.
* **Equivalent HGVS spellings share a row's verdict.** An indel in a repeat can be written at
  several positions; left-normalization proves they are the same variant, and the verdict is
  recorded against one representative spelling, so the other spellings carry read lists but no
  verdict (the cell that builds the maps prints how many were folded).
* **A few HGVS types have no VCF form here** (`dup`, `delins`, `inv`). They are absent from both
  sides of the benchmark, so they carry read lists but never a verdict.

In [15]:
df.tail()

,hgvsg,reads_mapped_file1_denovo,reads_mapped_file2_denovo,any_read_mapped_denovo,reads_mapped_file1_cosmic,reads_mapped_file2_cosmic,any_read_mapped_cosmic,tp_cosmic,fp_cosmic,fn_cosmic,tp_records_cosmic,fp_records_cosmic,fn_records_cosmic
1555944,15:g.90794315G>A,[],[],False,[],[],False,False,False,False,0,0,0
1555945,3:g.47106015C>A,[],[],False,[],[],False,False,False,False,0,0,0
1555946,15:g.90084282G>A,[],[],False,[],[],False,False,False,False,0,0,0
1555947,15:g.90085034C>T,[],[],False,[],[],False,False,False,False,0,0,0
1555948,15:g.90090649G>T,[],[],False,[],[],False,False,False,False,0,0,0


In [16]:
df.loc[df["fp_cosmic"]]

,hgvsg,reads_mapped_file1_denovo,reads_mapped_file2_denovo,any_read_mapped_denovo,reads_mapped_file1_cosmic,reads_mapped_file2_cosmic,any_read_mapped_cosmic,tp_cosmic,fp_cosmic,fn_cosmic,tp_records_cosmic,fp_records_cosmic,fn_records_cosmic
65843,20:g.45325634_45325635insT,"[642405, 2618903, 4613879, 5147180, 5173035, 5...","[138271, 904911, 1026256, 1312825, 2370439, 28...",True,"[5147180, 5976295]","[2370439, 2840730, 3247075, 3293881, 4318141]",True,True,True,False,1,1,0
76897,20:g.51790963A>C,"[1418125, 2462649, 3327985, 3417246, 3874766, ...","[684861, 795086, 1488461, 1534512, 2493748, 26...",True,"[1418125, 2462649, 4237625]","[684861, 795086, 1488461, 2680820, 3232129, 36...",True,True,True,False,1,1,0
142496,20:g.37402536_37402537insTGGCC,[],[],False,"[2353023, 2762096, 3301367, 4258828, 6496131]","[1220501, 2226057, 3825761, 5442524]",True,False,True,False,0,1,0
188448,20:g.45333956G>A,[],[],False,"[843250, 2884086, 6023586, 6087941]","[69723, 2653654, 3405670, 4366482, 4788428, 56...",True,False,True,False,0,1,0
188571,20:g.45333955_45333958del,[],[],False,"[843250, 2884086, 4314180, 6023586, 6087941]","[69723, 2653654, 3405670, 4366482, 4788428, 56...",True,False,True,False,0,1,0
266314,20:g.43112981G>A,[],[],False,"[99599, 302987, 586339, 1713469, 2757283, 2948...","[99599, 323952, 529811, 834754, 1504829, 16549...",True,False,True,False,0,1,0
268257,20:g.42625928_42625944del,[],[],False,"[115241, 1695533, 2239724, 2881579, 4833027, 6...","[220976, 568528, 1353570, 3615970, 5327117, 56...",True,False,True,False,0,1,0
268640,20:g.42627768_42627769insG,[],[],False,"[1095730, 1867646, 3191806, 3643501, 3899748, ...","[757635, 3432391, 4548490]",True,False,True,False,0,1,0
271625,20:g.42481783C>A,[],[],False,"[255885, 313686, 1669681, 2579777, 3241238, 43...",[10303],True,False,True,False,0,1,0
275713,20:g.42862010A>C,[],[],False,"[1109921, 5771409]","[1743383, 2443399, 2697276]",True,False,True,False,0,1,0


In [17]:
df.loc[df["fn_cosmic"]]

,hgvsg,reads_mapped_file1_denovo,reads_mapped_file2_denovo,any_read_mapped_denovo,reads_mapped_file1_cosmic,reads_mapped_file2_cosmic,any_read_mapped_cosmic,tp_cosmic,fp_cosmic,fn_cosmic,tp_records_cosmic,fp_records_cosmic,fn_records_cosmic
3842,20:g.1914729G>C,[],[],False,[],[],False,False,False,True,0,0,1
3843,20:g.1914731T>C,[],[],False,[],[],False,False,False,True,0,0,1
3853,20:g.1914955C>T,[],[],False,[],[],False,False,False,True,0,0,1
3854,20:g.1914965G>A,[],[],False,[],[],False,False,False,True,0,0,1
3855,20:g.1914975C>A,[],[],False,[],[],False,False,False,True,0,0,1
3856,20:g.1914984T>C,[],[],False,[],[],False,False,False,True,0,0,1
3857,20:g.1914998T>C,[],[],False,[],[],False,False,False,True,0,0,1
3858,20:g.1915009T>C,[],[],False,[],[],False,False,False,True,0,0,1
3859,20:g.1915012A>T,[],[],False,[],[],False,False,False,True,0,0,1
3860,20:g.1915021A>C,[],[],False,[],[],False,False,False,True,0,0,1


In [18]:
import pyfastx

vcrs_fa = pyfastx.Fasta("data/na12878_chr20/varseek_ref_out_cosmic/vcrs.fa")
r1 = pyfastx.Fastq("data/na12878_chr20/reads/NA12878_chr20_R1.fastq.gz")
r2 = pyfastx.Fastq("data/na12878_chr20/reads/NA12878_chr20_R2.fastq.gz")
ref_fa = pyfastx.Fasta(sequences)

# vcrs_fa["20:g.1915196T>C"].seq   # VCRS sequence by HGVS header
# r1[1132].seq                # read sequence by 0-based position — the BUS flag column

def has_kmer_overlap(seq1: str, seq2: str, k: int) -> bool:
    """True if seq1 and seq2 share a k-mer in either orientation (kb ran --strand unstranded)."""
    if k <= 0:
        raise ValueError("k must be positive")
    if len(seq1) < k or len(seq2) < k:
        return False
    rc = seq1.translate(str.maketrans("ACGTN", "TGCAN"))[::-1]
    kmers = {seq1[i:i + k] for i in range(len(seq1) - k + 1)}
    kmers |= {rc[i:i + k] for i in range(len(rc) - k + 1)}
    return any(seq2[i:i + k] in kmers for i in range(len(seq2) - k + 1))

In [19]:
variant = "20:g.43112981G>A"

In [20]:
reads_mapped_file1_cosmic, reads_mapped_file2_cosmic = df.loc[df["hgvsg"] == variant, ["reads_mapped_file1_cosmic", "reads_mapped_file2_cosmic"]].values[0]
print(reads_mapped_file1_cosmic)
print(reads_mapped_file2_cosmic)

[99599, 302987, 586339, 1713469, 2757283, 2948932, 3123781, 3472646, 3894066, 4166933, 4167807, 5919144, 6323503]
[99599, 323952, 529811, 834754, 1504829, 1654930, 2578898, 2968895, 2991791, 3086676, 3113204, 3532852, 3806824, 3851403, 4019715, 4624245, 5038456, 5802418, 6109471, 6419997, 6473564]


In [21]:
vcrs_header = !grep "{variant}" /home/jrich/Desktop/varseek-examples/data/na12878_chr20/varseek_ref_out_cosmic/vcrs.fa
vcrs_header = vcrs_header[0].split()[0][1:]
vcrs_position = int(variant.split("g.")[1][:-3])
vcrs_header

'20:g.43112981G>A'

In [22]:
vcrs_seq = vcrs_fa[vcrs_header].seq
print(vcrs_seq)
print(pyfastx.reverse_complement(vcrs_seq))
print(" "*40 + "*")

TCCTGGGGTTTTGCATGTAACTCATTCATGATATAAACGTATCTCCTGGGGTTTTGCATGTAACTCATTCATGATATAAAC
GTTTATATCATGAATGAGTTACATGCAAAACCCCAGGAGATACGTTTATATCATGAATGAGTTACATGCAAAACCCCAGGA
                                        *


In [23]:
read_idx = 0
read_seq = r2[reads_mapped_file2_cosmic[read_idx]].seq
print(read_seq)
# print(pyfastx.reverse_complement(read_seq))

TTAATTAAAAGAAGGAGACACGTTTATATCATGAATGAGTTACATGCAAAACCCCAGGAGGTACGTTTATATCATGAATGAGTTACATGCAAAACCCCAGGAGATACGTTTATATCATGAATGAGTTACATGCAAAACCCCAGGAGAT


In [24]:
# has_kmer_overlap(vcrs_seq, read_seq, k)

### Locating a read (and its k-mers) in the reference genome, quickly

`find_kmers` above is exact but scans all 3.1 Gbp of the genome once **per k-mer** in Python, so a
single 148 bp read costs ~200 passes over the genome — hours. The same question answered against an
FM-index takes **~0.2 s**, and `bowtie2` is already a varseek dependency: `vk ref` uses it to decide
which VCRSs also align to the normal reference (the d-list).

Two things are asked of the index below:

* **where the read maps** — `--local`, so a read that only partly matches the reference (soft
  clipping) still reports where its matching part sits;
* **where each of the read's k-mers occurs, exactly** — `--end-to-end --score-min L,0,0 -N 0`,
  which reports only perfect, full-length occurrences of the k-mer, i.e. exactly what `find_kmers`
  looks for.

Note the reference: `data/reference/genome_bowtie2_index` was built by `vk ref` from **T2T-CHM13**,
a different assembly from the GRCh38 everything else here is called against, so the cell below uses
a GRCh38 index (falling back to the chr20-only one if the genome index has not been built).

In [25]:
BOWTIE2_INDEXES = {
    "grch38": os.path.join(reference_dir, "bowtie2_grch38", "grch38"),
    "chr20": os.path.join(reference_dir, "bowtie2_chr20", "chr20"),
}
genome_index = next(path for path in BOWTIE2_INDEXES.values() if os.path.exists(path + ".1.bt2")
                    or os.path.exists(path + ".1.bt2l"))
print("bowtie2 index:", genome_index)


def bowtie2_locate(seqs, index=None, max_alignments=5, exact=False, threads=8):
    """Where do these sequences occur in the reference? One row per reported alignment.

    seqs    : {name: sequence}
    exact   : True  -> end-to-end, zero mismatches (an occurrence of the sequence verbatim)
              False -> local alignment (where the best-matching part of the sequence sits)
    Only the first `max_alignments` alignments per query are reported; `capped` marks the
    queries that hit that ceiling and therefore occur in at least that many places.
    """
    index = index or genome_index
    fasta = "".join(f">{name}\n{seq}\n" for name, seq in seqs.items())
    cmd = ["bowtie2", "-x", index, "-f", "-U", "-", "--no-unal", "--no-hd",
           "-p", str(threads), "--xeq"]
    cmd += ["-k", "5"]
    cmd += (["--end-to-end", "--score-min", "L,0,0", "-N", "0", "-L", "20", "-i", "C,1,0"]
            if exact else ["--local"])
    done = subprocess.run(cmd, input=fasta, capture_output=True, text=True, check=True)

    rows = []
    for line in done.stdout.splitlines():
        f = line.split("\t")
        if len(f) < 11:
            continue
        tags = dict(t.split(":", 2)[::2] for t in f[11:] if ":" in t)
        rows.append({"query": f[0], "chrom": f[2], "pos": int(f[3]),
                     "strand": "-" if int(f[1]) & 16 else "+", "mapq": int(f[4]),
                     "cigar": f[5], "mismatches": int(tags.get("NM", -1)),
                     "score": int(tags.get("AS", 0))})
    hits = pd.DataFrame(rows, columns=["query", "chrom", "pos", "strand", "mapq", "cigar",
                                       "mismatches", "score"])
    if len(hits):
        hits["capped"] = hits["query"].map(hits["query"].value_counts()) >= max_alignments
    return hits

bowtie2 index: data/reference/bowtie2_grch38/grch38


In [26]:
# # Where does the read itself come from, and where does the VCRS sequence occur?
# read_idx = 0
# read_seq = r2[reads_mapped_file2_cosmic[read_idx]].seq
# read_seq

placement = bowtie2_locate({"read": read_seq})
read_map_pos = placement["pos"].iloc[0] - 1
display(placement)

,query,chrom,pos,strand,mapq,cigar,mismatches,score,capped
0,read,20,43113024,-,32,148=,0,296,True
1,read,20,43112982,-,255,1S86=1X41=19S,1,248,True
2,read,20,43112938,-,255,43=1X43=1X41=1X6=12S,3,248,True
3,read,20,43112852,-,255,5=1X54=1X26=1X48=12S,3,248,True
4,read,20,43112895,-,255,17=1X68=2X41=1X6=12S,4,240,True


In [27]:
print(ref_fa["20"].seq[read_map_pos - 1:read_map_pos + len(read_seq) - 1])
print(pyfastx.reverse_complement(ref_fa["20"].seq[read_map_pos - 1:read_map_pos + len(read_seq) - 1]))

TATCTCCTGGGGTTTTGCATGTAACTCATTCATGATATAAACGTATCTCCTGGGGTTTTGCATGTAACTCATTCATGATATAAACGTACCTCCTGGGGTTTTGCATGTAACTCATTCATGATATAAACGTGTCTCCTTCTTTTAATTA
TAATTAAAAGAAGGAGACACGTTTATATCATGAATGAGTTACATGCAAAACCCCAGGAGGTACGTTTATATCATGAATGAGTTACATGCAAAACCCCAGGAGATACGTTTATATCATGAATGAGTTACATGCAAAACCCCAGGAGATA


In [28]:
print(ref_fa["20"].seq[vcrs_position-k:vcrs_position+k])

TAAACGTATCTCCTGGGGTTTTGCATGTAACTCATTCATGATATAAACGTGTCTCCTGGGGTTTTGCATGTAACTCATTCATGATATAAACGTATCTCCTGG


In [57]:
def count_unique_kmers(seq, k, max_bases_left=None, max_bases_right=None):
    """
    Count the number of unique k-mers in a sequence.

    Parameters
    ----------
    seq : str
        Input nucleotide sequence.
    k : int
        k-mer length.
    max_bases_left : int or None, default=None
        Maximum number of bases to keep to the left of the center base.
        If None, keep the entire left side.
    max_bases_right : int or None, default=None
        Maximum number of bases to keep to the right of the center base.
        If None, keep the entire right side.

    Returns
    -------
    int
        Number of unique k-mers.
    """
    seq = seq.upper()

    if k <= 0:
        raise ValueError("k must be positive")

    if len(seq) < k:
        return 0

    center = len(seq) // 2

    start = 0 if max_bases_left is None else max(0, center - max_bases_left)
    end = len(seq) if max_bases_right is None else min(len(seq), center + max_bases_right + 1)

    seq = seq[start:end]

    if len(seq) < k:
        return 0

    return len({seq[i:i+k] for i in range(len(seq) - k + 1)})

In [48]:
header_map = {}

with open("/home/jrich/Desktop/varseek-examples/data/na12878_chr20/varseek_ref_out_cosmic/vcrs.fa") as f:
    for line in f:
        if line.startswith(">"):
            vcrs_header = line[1:].split()[0]      # e.g. 20:g.51790963A>C;20:g.51712963A>CHROMOSOMES
            variant_headers = vcrs_header.split(";")
            for variant_header in variant_headers:
                header_map[variant_header] = vcrs_header

df["vcrs_header"] = df["hgvsg"].map(header_map)

In [ ]:
mask = df[["tp_cosmic", "fp_cosmic", "fn_cosmic"]].any(axis=1)

configs = [
    ("unique_doublets", 2, None, None),
    ("unique_triplets", 3, None, None),

    ("unique_singlets_6_right", 1, 0, 6),
    ("unique_doublets_6_right", 2, 0, 6),
    ("unique_triplets_6_right", 3, 0, 6),

    ("unique_singlets_6_left", 1, 6, 0),
    ("unique_doublets_6_left", 2, 6, 0),
    ("unique_triplets_6_left", 3, 6, 0),

    ("unique_singlets_6_left_or_right", 1, 6, 6),
    ("unique_doublets_6_left_or_right", 2, 6, 6),
    ("unique_triplets_6_left_or_right", 3, 6, 6),
]

for col, k, max_left, max_right in configs:
    df.loc[mask, col] = (
        df.loc[mask, "vcrs_header"]
        .map(
            lambda h: (
                count_unique_kmers(
                    str(vcrs_fa[h].seq),
                    k=k,
                    max_bases_left=max_left,
                    max_bases_right=max_right,
                )
                if h in vcrs_fa
                else np.nan
            )
        )
    )

In [62]:
df.loc[df[["tp_cosmic", "fp_cosmic", "fn_cosmic"]].any(axis=1)]

,hgvsg,reads_mapped_file1_denovo,reads_mapped_file2_denovo,any_read_mapped_denovo,reads_mapped_file1_cosmic,reads_mapped_file2_cosmic,any_read_mapped_cosmic,tp_cosmic,fp_cosmic,fn_cosmic,...,unique_doublets,unique_singlets_6_right,unique_doublets_6_right,unique_triplets_6_right,unique_singlets_6_left,unique_doublets_6_left,unique_triplets_6_left,unique_singlets_6_left_or_right,unique_doublets_6_left_or_right,unique_triplets_6_left_or_right
3842,20:g.1914729G>C,[],[],False,[],[],False,False,False,True,...,15.0,4.0,15.0,30.0,4.0,14.0,29.0,4.0,7.0,10.0
3843,20:g.1914731T>C,[],[],False,[],[],False,False,False,True,...,15.0,4.0,15.0,29.0,4.0,14.0,30.0,4.0,8.0,10.0
3853,20:g.1914955C>T,[],[],False,[],[],False,False,False,True,...,15.0,4.0,14.0,30.0,4.0,15.0,32.0,3.0,9.0,11.0
3854,20:g.1914965G>A,[],[],False,[],[],False,False,False,True,...,15.0,4.0,14.0,29.0,4.0,15.0,33.0,4.0,8.0,10.0
3855,20:g.1914975C>A,[],[],False,[],[],False,False,False,True,...,15.0,4.0,14.0,27.0,4.0,15.0,34.0,4.0,8.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377090,20:g.51398583del,[],[],False,"[4312696, 6091709]",[5225577],True,True,False,False,...,15.0,4.0,13.0,22.0,4.0,14.0,22.0,2.0,2.0,2.0
1397244,20:g.42488963del,[],[],False,"[1396698, 3257662, 6349995]",[],True,True,False,False,...,13.0,4.0,11.0,21.0,3.0,6.0,11.0,2.0,2.0,2.0
1421114,20:g.51457635_51457636insGTGAGAGTCTAATCCAGCTGC...,[],[],False,"[253276, 296317, 783101, 1084181, 1436112, 177...","[1495565, 1781377, 3127657, 5718970, 6427327, ...",True,True,False,False,...,16.0,4.0,14.0,26.0,4.0,16.0,37.0,4.0,10.0,11.0
1497969,20:g.42795196del,[],[],False,"[3483455, 6261652]","[4813897, 5939431, 6400155]",True,True,False,False,...,15.0,4.0,11.0,19.0,4.0,13.0,21.0,2.0,3.0,3.0


In [63]:
df.loc[df[["fp_cosmic"]].any(axis=1)]

,hgvsg,reads_mapped_file1_denovo,reads_mapped_file2_denovo,any_read_mapped_denovo,reads_mapped_file1_cosmic,reads_mapped_file2_cosmic,any_read_mapped_cosmic,tp_cosmic,fp_cosmic,fn_cosmic,...,unique_doublets,unique_singlets_6_right,unique_doublets_6_right,unique_triplets_6_right,unique_singlets_6_left,unique_doublets_6_left,unique_triplets_6_left,unique_singlets_6_left_or_right,unique_doublets_6_left_or_right,unique_triplets_6_left_or_right
65843,20:g.45325634_45325635insT,"[642405, 2618903, 4613879, 5147180, 5173035, 5...","[138271, 904911, 1026256, 1312825, 2370439, 28...",True,"[5147180, 5976295]","[2370439, 2840730, 3247075, 3293881, 4318141]",True,True,True,False,...,16.0,4.0,12.0,21.0,4.0,14.0,28.0,3.0,5.0,7.0
76897,20:g.51790963A>C,"[1418125, 2462649, 3327985, 3417246, 3874766, ...","[684861, 795086, 1488461, 1534512, 2493748, 26...",True,"[1418125, 2462649, 4237625]","[684861, 795086, 1488461, 2680820, 3232129, 36...",True,True,True,False,...,15.0,4.0,14.0,33.0,4.0,15.0,32.0,4.0,10.0,11.0
142496,20:g.37402536_37402537insTGGCC,[],[],False,"[2353023, 2762096, 3301367, 4258828, 6496131]","[1220501, 2226057, 3825761, 5442524]",True,False,True,False,...,16.0,4.0,14.0,31.0,4.0,16.0,34.0,4.0,9.0,10.0
188448,20:g.45333956G>A,[],[],False,"[843250, 2884086, 6023586, 6087941]","[69723, 2653654, 3405670, 4366482, 4788428, 56...",True,False,True,False,...,14.0,4.0,11.0,20.0,4.0,14.0,24.0,2.0,3.0,4.0
188571,20:g.45333955_45333958del,[],[],False,"[843250, 2884086, 4314180, 6023586, 6087941]","[69723, 2653654, 3405670, 4366482, 4788428, 56...",True,False,True,False,...,14.0,4.0,11.0,20.0,4.0,14.0,24.0,2.0,3.0,4.0
266314,20:g.43112981G>A,[],[],False,"[99599, 302987, 586339, 1713469, 2757283, 2948...","[99599, 323952, 529811, 834754, 1504829, 16549...",True,False,True,False,...,15.0,4.0,15.0,31.0,4.0,15.0,31.0,4.0,9.0,11.0
268257,20:g.42625928_42625944del,[],[],False,"[115241, 1695533, 2239724, 2881579, 4833027, 6...","[220976, 568528, 1353570, 3615970, 5327117, 56...",True,False,True,False,...,14.0,4.0,13.0,25.0,4.0,13.0,24.0,4.0,11.0,11.0
268640,20:g.42627768_42627769insG,[],[],False,"[1095730, 1867646, 3191806, 3643501, 3899748, ...","[757635, 3432391, 4548490]",True,False,True,False,...,15.0,4.0,13.0,25.0,4.0,14.0,34.0,4.0,6.0,8.0
271625,20:g.42481783C>A,[],[],False,"[255885, 313686, 1669681, 2579777, 3241238, 43...",[10303],True,False,True,False,...,14.0,4.0,13.0,21.0,4.0,9.0,15.0,2.0,3.0,5.0
275713,20:g.42862010A>C,[],[],False,"[1109921, 5771409]","[1743383, 2443399, 2697276]",True,False,True,False,...,16.0,4.0,14.0,31.0,4.0,15.0,29.0,3.0,6.0,10.0


In [64]:
df.loc[df[["fn_cosmic"]].any(axis=1)]

,hgvsg,reads_mapped_file1_denovo,reads_mapped_file2_denovo,any_read_mapped_denovo,reads_mapped_file1_cosmic,reads_mapped_file2_cosmic,any_read_mapped_cosmic,tp_cosmic,fp_cosmic,fn_cosmic,...,unique_doublets,unique_singlets_6_right,unique_doublets_6_right,unique_triplets_6_right,unique_singlets_6_left,unique_doublets_6_left,unique_triplets_6_left,unique_singlets_6_left_or_right,unique_doublets_6_left_or_right,unique_triplets_6_left_or_right
3842,20:g.1914729G>C,[],[],False,[],[],False,False,False,True,...,15.0,4.0,15.0,30.0,4.0,14.0,29.0,4.0,7.0,10.0
3843,20:g.1914731T>C,[],[],False,[],[],False,False,False,True,...,15.0,4.0,15.0,29.0,4.0,14.0,30.0,4.0,8.0,10.0
3853,20:g.1914955C>T,[],[],False,[],[],False,False,False,True,...,15.0,4.0,14.0,30.0,4.0,15.0,32.0,3.0,9.0,11.0
3854,20:g.1914965G>A,[],[],False,[],[],False,False,False,True,...,15.0,4.0,14.0,29.0,4.0,15.0,33.0,4.0,8.0,10.0
3855,20:g.1914975C>A,[],[],False,[],[],False,False,False,True,...,15.0,4.0,14.0,27.0,4.0,15.0,34.0,4.0,8.0,10.0
3856,20:g.1914984T>C,[],[],False,[],[],False,False,False,True,...,16.0,4.0,15.0,32.0,4.0,16.0,37.0,4.0,8.0,11.0
3857,20:g.1914998T>C,[],[],False,[],[],False,False,False,True,...,16.0,4.0,13.0,29.0,4.0,16.0,35.0,3.0,7.0,9.0
3858,20:g.1915009T>C,[],[],False,[],[],False,False,False,True,...,16.0,4.0,14.0,34.0,4.0,16.0,33.0,4.0,8.0,11.0
3859,20:g.1915012A>T,[],[],False,[],[],False,False,False,True,...,16.0,4.0,15.0,32.0,4.0,16.0,32.0,4.0,9.0,11.0
3860,20:g.1915021A>C,[],[],False,[],[],False,False,False,True,...,16.0,4.0,15.0,34.0,4.0,15.0,30.0,4.0,10.0,11.0
